In [ ]:
!pip -q install transformers sentencepiece sacremoses sacrebleu

In [ ]:
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device).eval()
    return tok, model

def translate(texts, tok, model, prefix="", beams=4):
    batch = tok([prefix + t for t in texts], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**batch, num_beams=beams, max_new_tokens=128)
    return tok.batch_decode(out, skip_special_tokens=True)

sentences = [
    "The weather is very nice today.",
    "I am learning natural language processing at college.",
    "Machine translation helps people communicate across languages.",
]

In [ ]:
# ---------- MarianMT: English -> Hindi ----------
tok_hi, m_hi = load("Helsinki-NLP/opus-mt-en-hi")
hi = translate(sentences, tok_hi, m_hi)
for s, t in zip(sentences, hi): print(f"EN: {s}\nHI: {t}\n")

In [ ]:
# ---------- MarianMT: English -> French ----------
tok_fr, m_fr = load("Helsinki-NLP/opus-mt-en-fr")
fr = translate(sentences, tok_fr, m_fr)
for s, t in zip(sentences, fr): print(f"EN: {s}\nFR: {t}\n")

In [ ]:
# ---------- T5-small: English -> German / French (task prefix) ----------
tok_t5, m_t5 = load("t5-small")
de_t5 = translate(sentences, tok_t5, m_t5, prefix="translate English to German: ")
fr_t5 = translate(sentences, tok_t5, m_t5, prefix="translate English to French: ")
for s, d, f in zip(sentences, de_t5, fr_t5):
    print(f"EN: {s}\nDE (T5): {d}\nFR (T5): {f}\n")

In [ ]:
# ---------- Evaluation with BLEU (English -> French reference translations) ----------
refs_fr = [
    "Il fait très beau aujourd'hui.",
    "J'apprends le traitement automatique du langage naturel à l'université.",
    "La traduction automatique aide les gens à communiquer entre les langues.",
]
bleu_marian = sacrebleu.corpus_bleu(fr, [refs_fr]).score
bleu_t5 = sacrebleu.corpus_bleu(fr_t5, [refs_fr]).score
display(pd.DataFrame({"model": ["MarianMT (opus-mt-en-fr)", "T5-small"], "BLEU": [round(bleu_marian, 2), round(bleu_t5, 2)]}))

In [ ]:
# Translate your own sentence
my_text = ["Students should practice coding every day."]
print("Hindi :", translate(my_text, tok_hi, m_hi)[0])
print("French:", translate(my_text, tok_fr, m_fr)[0])